# Reproduce real numeric results — CPU only

This notebook reads **only existing sanitized real numeric scores** and an optional sanitized planned-run manifest. It imports no provider, opens no transcript, and makes no API call. It recalculates paired metrics, 2,000 family bootstrap draws, figures and end-to-end cost. Missing results produce **Experiment not executed**, never plausible replacement scores.

Set `CONTEXT_AUDIT_SCORES` to analyze another sanitized real score CSV; the default is `results/public_scores.csv`. An adjacent `run_manifest.json` verifies planned coverage, including entirely missing pairs. Without a planned manifest, observed-only AUROC is descriptive and cannot pass the primary coverage gate. Synthetic fixtures are rejected.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "research_plan.md").exists()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required"
print("Python:", sys.version.split()[0])
print("Repository found; no model API request has been made.")

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

from context_audit.metrics import validate_rows
from context_audit.reporting import generate_report

SCORES = Path(os.environ.get("CONTEXT_AUDIT_SCORES", str(ROOT / "results" / "public_scores.csv")))
MANIFEST = SCORES.with_name("run_manifest.json")
OUTPUT = ROOT / "results" / "reproduced"
SEED = 20260905
BOOTSTRAP_SAMPLES = 2000
rows = pd.read_csv(SCORES) if SCORES.exists() and SCORES.stat().st_size else pd.DataFrame()
rows = validate_rows(rows, allow_partial_pairs=MANIFEST.exists())
if rows.empty:
    metrics = None
    print(
        "Experiment not executed: no real numeric scores are available. No results were fabricated."
    )
else:
    if set(rows["data_origin"]) != {"sleight_bench"}:
        raise ValueError("This notebook rejects synthetic fixtures and mixed origins.")
    metrics = generate_report(
        SCORES,
        OUTPUT,
        n_bootstrap=BOOTSTRAP_SAMPLES,
        seed=SEED,
        manifest_path=MANIFEST if MANIFEST.exists() else None,
    )
    print("Offline reproduction complete; no API call was made.")
    display(Markdown((OUTPUT / "findings.md").read_text()))

## Primary AUROC and paired uncertainty

The common cohort must contain both benign and malicious members, with valid scores in all four conditions. Bootstrap samples whole families and retains scenarios, members, conditions and intentional repetitions together. Repeated API calls are not new independent cases. The primary effect is `AUROC(structured_summary) - AUROC(free_summary)`.

The proposed coverage threshold is 95%. A failed or unverifiable gate prevents an unqualified principal conclusion. Conditional success selection remains a limitation even when coverage passes. Confidence intervals characterize these sampled families, not all deployments.

In [ ]:
if metrics is not None:
    primary = metrics["primary"]
    display(
        pd.DataFrame(
            [
                {
                    "condition": name,
                    **values["auroc"],
                    "valid": values["n_valid"],
                    "expected": values["n_expected"],
                    "missing": values["n_missing"],
                }
                for name, values in metrics["conditions"].items()
            ]
        )
    )
    print("Delta AUROC:", primary["delta_auroc"])
    print(
        "Common observations:", primary["n_observations"], "/", primary["n_expected_observations"]
    )
    print("Coverage gate passed:", primary["quality_gate_passed"])
    display(Image(filename=str(OUTPUT / "figures" / "auroc.png")))
else:
    print("AUROC and confidence intervals unavailable until a real experiment is executed.")

## Fixed threshold, failures and denominators

At score ≥ 50, report TPR, FPR and balanced accuracy with their positive/negative denominators. Failure scores remain null. Operational end-to-end rates escalate recorded failures to review; missing records have no imputed score or decision. Threshold 50 is an exploratory convention, not a validated operating point. With few distinct benign trajectories, 1% FPR is not a defensible guarantee.

In [ ]:
if metrics is not None:
    rates = []
    for condition, values in metrics["conditions"].items():
        for analysis_set in ["threshold_valid", "threshold_end_to_end"]:
            rates.append({"condition": condition, "set": analysis_set, **values[analysis_set]})
    display(pd.DataFrame(rates))
    print(
        "Minimum distinct-benign false-positive step:",
        metrics["minimum_unique_false_positive_step"],
    )
    display(Image(filename=str(OUTPUT / "figures" / "threshold_rates.png")))
else:
    print("Threshold rates unavailable; no real decision denominators exist.")

## Cost, context length and qualitative worksheet

Total generation cost includes summarization, monitoring and retries. Unknown billed usage is flagged as a conservative upper bound, not silently declared exact. Actual provider-cache charges are retained; offline cached reanalysis adds no generation cost. A hypothetical cache-free rerun is not fabricated. Compare realized sizes because a shared ceiling is not a matched realized length.

The qualitative worksheet selects at most six discordances and six controls by a fixed seeded hash, using opaque IDs only. Copy it into `runs/private/` before adding source event evidence, category, certainty or reviewer notes. No review finding is generated automatically. Selected discordances do not estimate prevalence or prove the cause of an error.

In [ ]:
if metrics is not None:
    display(
        pd.DataFrame(
            [
                {
                    "condition": condition,
                    "summary_usd": values["summary_cost_usd"],
                    "monitor_usd": values["monitor_cost_usd"],
                    "total_usd": values["cost_usd"],
                    "upper_bound_rows": values["n_cost_upper_bound"],
                    "mean_size_ratio": values["realized_token_ratio"]["mean"],
                    "mean_latency_seconds": values["latency_seconds"]["mean"],
                }
                for condition, values in metrics["conditions"].items()
            ]
        )
    )
    display(Image(filename=str(OUTPUT / "figures" / "cost_performance.png")))
    display(pd.read_csv(OUTPUT / "qualitative_selection.csv").fillna(""))
    print("Cached offline reanalysis API cost: $0. No qualitative annotations have been inferred.")
else:
    print("Costs, figures and qualitative selections unavailable; experiment not executed.")

## Interpretation and reproducibility

Full is a reference within the visible-event view, not a guaranteed upper bound. The structured/free comparison measures the complete summarization intervention, not isolated JSON syntax. The benchmark is small and synthetic in construction; a real API evaluation on it does not establish production safety or exact reproduction of the SLEIGHT-Bench paper.

Keep this notebook's committed outputs empty. Public outputs may contain only opaque numeric tables, figures and reviewed nonsensitive interpretation. Preserve source canaries in private derived content and run the repository's content scan before an explicitly authorized publication. Record any test-informed protocol change as exploratory.